## Load Silver Data

In [0]:
from pyspark.sql.functions import (
    col, lag, avg, stddev, when, lit, abs as spark_abs,
    min as spark_min, max as spark_max, sum as spark_sum,
    round as spark_round, count, datediff, expr
)
from pyspark.sql.window import Window

stock_prices = spark.table("raghavan_trading_signals.silver.stock_prices")
macro_wide = spark.table("raghavan_trading_signals.silver.macro_daily_wide")

print(f"Stock prices: {stock_prices.count():,} rows")
print(f"Macro data: {macro_wide.count():,} rows")

## Daily Returns
- The most fundamental feature: how much did the stock move today?

In [0]:
price_window = Window.partitionBy("symbol").orderBy("trade_date")

features = (
    stock_prices
    .withColumn("prev_close", lag("close", 1).over(price_window))
    .withColumn("daily_return",
        spark_round((col("close") - col("prev_close")) / col("prev_close"), 6)
    )
    .withColumn("daily_return_pct",
        spark_round(col("daily_return") * 100, 4)
    )
    .withColumn("log_return",
        spark_round(expr("ln(close / prev_close)"), 6)
    )
    # High-Low range as a volatility proxy
    .withColumn("daily_range",
        spark_round((col("high") - col("low")) / col("close"), 6)
    )
    # Gap: how much did the stock gap up/down from yesterday's close?
    .withColumn("overnight_gap",
        spark_round((col("open") - col("prev_close")) / col("prev_close"), 6)
    )
)

## Moving Averages
- Moving averages smooth out noise and reveal trends.
- If price > MA, the stock is in an uptrend (and vice versa).
- Price relative to moving average (normalized, so it's comparable across stocks)

In [0]:
for period in [7, 14, 21, 50, 100, 200]:
    ma_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"sma_{period}",
        spark_round(avg("close").over(ma_window), 4)
    )
    features = features.withColumn(
        f"price_vs_sma_{period}",
        spark_round((col("close") - col(f"sma_{period}")) / col(f"sma_{period}"), 6)
    )

## Volatility (Rolling Standard Deviation)
- Volatility = how much the stock bounces around.
- Higher volatility = more risk = wider prediction intervals.

In [0]:
for period in [10, 20, 60]:
    vol_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"volatility_{period}d",
        spark_round(stddev("daily_return").over(vol_window), 6)
    )

## RSI (Relative Strength Index)
- RSI measures momentum: 0-100 scale.
- Below 30 = "oversold" (might bounce up), above 70 = "overbought" (might drop).
- RSI = 100 - (100 / (1 + RS)), where RS = avg_gain / avg_loss over N periods.

In [0]:
rsi_period = 14
rsi_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
    -(rsi_period - 1), Window.currentRow
)

features = (
    features
    .withColumn("price_change", col("close") - col("prev_close"))
    .withColumn("gain", when(col("price_change") > 0, col("price_change")).otherwise(0))
    .withColumn("loss", when(col("price_change") < 0, spark_abs(col("price_change"))).otherwise(0))
    .withColumn("avg_gain", avg("gain").over(rsi_window))
    .withColumn("avg_loss", avg("loss").over(rsi_window))
    .withColumn("rs",
        when(col("avg_loss") == 0, lit(100))
        .otherwise(col("avg_gain") / col("avg_loss"))
    )
    .withColumn("rsi_14",
        spark_round(lit(100) - (lit(100) / (lit(1) + col("rs"))), 4)
    )
    .drop("price_change", "gain", "loss", "avg_gain", "avg_loss", "rs")
)

## MACD (Moving Average Convergence Divergence)
- MACD reveals changes in trend strength, direction, momentum, and duration.
- MACD Line = 12-day EMA - 26-day EMA
- Signal Line = 9-day EMA of MACD Line
- Histogram = MACD Line - Signal Line (positive = bullish momentum)
- Spark doesn't have a built-in EMA, so we approximate with SMA-based calculation.
- For a production system, you'd use a UDF or Pandas UDF for true EMA.

In [0]:
ema_12_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-11, Window.currentRow)
ema_26_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-25, Window.currentRow)

features = (
    features
    .withColumn("ema_12_approx", spark_round(avg("close").over(ema_12_window), 4))
    .withColumn("ema_26_approx", spark_round(avg("close").over(ema_26_window), 4))
    .withColumn("macd_line",
        spark_round(col("ema_12_approx") - col("ema_26_approx"), 4)
    )
)

macd_signal_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-8, Window.currentRow)
features = features.withColumn(
    "macd_signal",
    spark_round(avg("macd_line").over(macd_signal_window), 4)
)
features = features.withColumn(
    "macd_histogram",
    spark_round(col("macd_line") - col("macd_signal"), 4)
)

features = features.drop("ema_12_approx", "ema_26_approx")

## Bollinger Bands
- Bollinger Bands = SMA(20) ± 2 * StdDev(20).
- When price touches upper band = potentially overbought.
- When price touches lower band = potentially oversold.
- Band width indicates volatility.

In [0]:
bb_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-19, Window.currentRow)

features = (
    features
    .withColumn("bb_middle", spark_round(avg("close").over(bb_window), 4))
    .withColumn("bb_std", stddev("close").over(bb_window))
    .withColumn("bb_upper", spark_round(col("bb_middle") + (lit(2) * col("bb_std")), 4))
    .withColumn("bb_lower", spark_round(col("bb_middle") - (lit(2) * col("bb_std")), 4))
    .withColumn("bb_width",
        spark_round((col("bb_upper") - col("bb_lower")) / col("bb_middle"), 6)
    )
    .withColumn("bb_position",
        spark_round(
            (col("close") - col("bb_lower")) / (col("bb_upper") - col("bb_lower")),
            4
        )
    )
    .drop("bb_std")
)

## Volume Features

- Volume confirms price movements. A price spike on high volume is more significant.
- Volume ratio: today's volume vs 20-day average

In [0]:
for period in [10, 20, 50]:
    vol_ma_window = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(
        -(period - 1), Window.currentRow
    )
    features = features.withColumn(
        f"volume_sma_{period}",
        spark_round(avg("volume").over(vol_ma_window), 0)
    )

features = features.withColumn(
    "volume_ratio",
    spark_round(col("volume") / col("volume_sma_20"), 4)
)

## Join Macro Data

In [0]:
features = features.join(
    macro_wide,
    features["trade_date"] == macro_wide["indicator_date"],
    "left"
).drop("indicator_date")

# Compute yield curve slope (10Y - 2Y) — a key recession predictor
features = features.withColumn(
    "yield_curve_slope",
    spark_round(col("treasury_10y") - col("treasury_2y"), 4)
)

## Create the Target Label
- This is what the ML model will predict:
- Did the stock go UP or DOWN the NEXT day?

In [0]:
features = features.withColumn(
    "next_day_return",
    spark_round(lag("daily_return", -1).over(price_window), 6)
)
features = features.withColumn(
    "next_day_direction",
    when(col("next_day_return") > 0, lit(1)).otherwise(lit(0))
)

## Drop Warmup Rows and Write to Gold
- The first ~200 rows per stock have incomplete moving averages (not enough history).
- Drop them so the ML model doesn't train on garbage.

In [0]:
features_clean = (
    features
    .filter(col("sma_200").isNotNull())
    .filter(col("next_day_direction").isNotNull())
    .orderBy("symbol", "trade_date")
)

print(f"Gold feature table: {features_clean.count():,} rows")
print(f"Columns: {len(features_clean.columns)}")

# Write to Gold as a managed Delta table
(
    features_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("raghavan_trading_signals.gold.daily_features")
)

print("Gold table written successfully!")

## Validation

In [0]:
gold_df = spark.table("raghavan_trading_signals.gold.daily_features")

print(f"Total rows: {gold_df.count():,}")
print(f"Total columns: {len(gold_df.columns)}")
print(f"Stocks: {gold_df.select('symbol').distinct().count()}")
print(f"Date range: {gold_df.agg({'trade_date': 'min'}).collect()[0][0]} to "
      f"{gold_df.agg({'trade_date': 'max'}).collect()[0][0]}")

# Show sample for one stock
display(
    gold_df
    .filter(col("symbol") == "AAPL")
    .orderBy(col("trade_date").desc())
    .select("symbol", "trade_date", "close", "rsi_14", "macd_line",
            "bb_position", "volatility_20d", "volume_ratio",
            "vix", "yield_curve_slope", "next_day_direction")
    .limit(10)
)